# ChronoPDE Week 4-5 GPU gates
Run the cells in order on a Colab T4 or A100. AR gates close before CT-FFT training.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('CHRONOPDE_REPOSITORY_URL') or input('Repository URL: ')
subprocess.run(['git', 'clone', repository_url, '/content/Chrono_pde'], check=True)
os.chdir('/content/Chrono_pde')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True)

In [ ]:
import hashlib
from pathlib import Path

import torch

DATA = Path('/content/drive/MyDrive/ChronoPDE/data/chronopde.h5')
assert DATA.stat().st_size == 2389261328
digest = hashlib.sha256()
with DATA.open('rb') as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        digest.update(block)
assert digest.hexdigest() == '907aa0d79e604e68ce2d4f5cccfd93ffc64eb68c473caf3edae4be438472caec'
assert torch.cuda.is_available(), 'Select a GPU runtime'
print(torch.cuda.get_device_name(), torch.__version__)

In [ ]:
for model_name in ('unet_ar', 'fno_ar'):
    command = [
        sys.executable, 'scripts/train.py', '--config', 'configs/project.yaml',
        '--model', model_name, '--regime', 'full', '--seed', '0',
        '--smoke-overfit', '--device', 'cuda', '--data-path', str(DATA),
    ]
    subprocess.run(command, check=True)

In [ ]:
for model_name in ('unet_ar', 'fno_ar'):
    command = [
        sys.executable, 'scripts/train.py', '--config', 'configs/project.yaml',
        '--model', model_name, '--regime', 'full', '--seed', '0',
        '--device', 'cuda', '--data-path', str(DATA), '--resume',
    ]
    subprocess.run(command, check=True)

In [ ]:
for model_name in ('unet_ar', 'fno_ar'):
    checkpoint = f'artifacts/runs/{model_name}-full-train-s0/best.pt'
    command = [
        sys.executable, 'scripts/evaluate.py', '--config', 'configs/project.yaml',
        '--model', model_name, '--experiment', 'id_rollout',
        '--checkpoint', checkpoint, '--device', 'cuda',
        '--data-path', str(DATA),
    ]
    subprocess.run(command, check=True)

## Week 5: continuous-time FFT
The smoke command must pass before the full resumable run starts.

In [ ]:
command = [
    sys.executable, 'scripts/train.py', '--config', 'configs/project.yaml',
    '--model', 'fno_ct', '--regime', 'full', '--seed', '0',
    '--smoke-overfit', '--device', 'cuda', '--data-path', str(DATA),
]
subprocess.run(command, check=True)

In [ ]:
command = [
    sys.executable, 'scripts/train.py', '--config', 'configs/project.yaml',
    '--model', 'fno_ct', '--regime', 'full', '--seed', '0',
    '--device', 'cuda', '--data-path', str(DATA), '--resume',
]
subprocess.run(command, check=True)

In [ ]:
checkpoint = 'artifacts/runs/fno_ct-full-train-s0/best.pt'
command = [
    sys.executable, 'scripts/evaluate.py', '--config', 'configs/project.yaml',
    '--model', 'fno_ct', '--experiment', 'id_rollout',
    '--checkpoint', checkpoint, '--device', 'cuda', '--data-path', str(DATA),
]
subprocess.run(command, check=True)